# 12 · The OKF agent as an A2A service

Stage 3, step 2. In `11` the agent lived **inside the notebook** (`OKFAgent.start()`): only that Python process could use it. Now it runs as its **own server**, `langgraph dev`, and anything that speaks **A2A** (agent-to-agent protocol) can talk to it over HTTP: this notebook, the router agent (step 14), other people's agents later.

```
 this notebook / router agent                 langgraph dev  (:2024)                  already running
 ───────────────────────────                 ───────────────────────────────          ─────────────────
 1. GET agent card  ───────────────────────►  "I'm okf_agent, I answer questions
                     ◄───────────────────────  about the OKF bundle, send to /a2a/…"
 2. POST /a2a/<id>  SendMessage "And GPT-3.5?"
    (+ contextId = which conversation) ────►  okf_agent graph:                    ──► MCP server (:8000)
                                               prepare → search → answer → check → log ──► Ollama (qwen)
                     ◄───────────────────────  task COMPLETED + answer             ──► Postgres (qa_log)
```

What changes compared to `11`:

| | `11` (in the notebook) | `12` (as a service) |
|---|---|---|
| who can use it | this Python process | anything that can send HTTP |
| conversation id | `thread_id="chat-1"` (you choose) | `contextId` (the **server** creates it, a UUID) |
| chat memory | Postgres (`PostgresSaver`) | the server's own memory, saved in `.langgraph_api/` |
| `qa_log` | Postgres | Postgres (same table, `thread_id` = `contextId`) |
| tracing | LangSmith | LangSmith (the server traces every run) |

The code: `okf_agent/server.py` (the graph the server loads), `langgraph.json` (tells `langgraph dev` where it is), `okf_agent/a2a_client.py` (a small A2A client).

## 0. Start the services

1. **Install** (once, in the `unsloth` env): `pip install -e ".[agent]"`. It now also installs `langgraph-cli[inmem]`, which gives the `langgraph dev` command.
2. **Start the backend**: in your `unsloth` terminal, in the OKF folder, run
   ```
   start_backend.bat
   ```
   It opens two windows: the **MCP server** (`:8000`) and the **agent server** `langgraph dev` (`:2024`). Leave both open. (Without the .bat: run `python -m okf_mcp.server` in one terminal and `langgraph dev --no-browser` in another.)
3. Postgres and Ollama running, as in `11`.

Wait until the agent window shows `API: http://127.0.0.1:2024`, then run the next cell.

In [1]:
%load_ext autoreload
%autoreload 2

import json
import os
import socket

import httpx
import pandas as pd
import psycopg

from okf_agent import AgentConfig
from okf_agent.a2a_client import A2AClient, langgraph_assistant_id, langgraph_card_url

AGENT_URL = os.getenv("OKF_AGENT_URL", "http://127.0.0.1:2024")     # where langgraph dev runs
config = AgentConfig.load()

def port_open(url):
    host, port = url.split("//")[1].split("/")[0].split(":")
    with socket.socket() as s:
        s.settimeout(2)
        return s.connect_ex((host, int(port))) == 0

print("MCP server up:  ", port_open(config.mcp_url), "  <- if False: python -m okf_mcp.server")
print("agent server up:", port_open(AGENT_URL), "  <- if False: langgraph dev --no-browser")
try:
    with psycopg.connect(config.postgres_url, connect_timeout=5) as c:
        print("Postgres ok:     True")
except Exception as exc:
    print("Postgres FAILED:", exc)

MCP server up:   True   <- if False: python -m okf_mcp.server
agent server up: True   <- if False: langgraph dev --no-browser
Postgres ok:     True


## 1. How `langgraph dev` finds the agent: `langgraph.json`

```json
{
  "dependencies": ["."],
  "graphs": {
    "okf_agent": {
      "path": "okf_agent.server:graph",
      "description": "Answers questions about the OKF bundle ... with [concept id] citations ..."
    }
  },
  "env": ".env"
}
```

- `graphs` → name → **where the graph object is** (`module:variable`). The server imports `okf_agent/server.py` and takes its `graph`.
- `description` → ends up on the **agent card**. A router reads it to decide whether this agent should get a question, so it's written for a reader that has never seen the project.
- `env` → the server loads `.env` (Postgres, LangSmith, …).

`okf_agent/server.py` builds **the same graph as `11`** with two differences:

```python
graph = build_graph(LazyTools(CONFIG.mcp_url),            # MCP tools load on the first question
                    traced_llm(make_llm(CONFIG.okf)),      # qwen, same settings as the evaluation
                    LazyLogConnection(CONFIG.postgres_url) # qa_log, opened on first use
                   ).compile()                              # no checkpointer: the server brings its own
```

- **No checkpointer.** The server keeps the memory itself, one thread per A2A conversation.
- **Nothing connects at import.** So the order you start things in doesn't matter: if the MCP server comes up after `langgraph dev`, the first question still works.

For each graph the server creates an **assistant** with a fixed UUID (derived from the name). A2A addresses agents by that UUID:

In [3]:
assistant_id = langgraph_assistant_id(AGENT_URL, "okf_agent")
print("assistant id:", assistant_id)      # always the same for the name "okf_agent"

assistant id: bb1780c6-cc9f-5dd1-a605-b5608e80a356


## 2. The agent card: the agent's "business card"

Every A2A agent publishes a JSON card at a well-known URL. It says **who** the agent is, **what** it can do (`skills`) and **where** to send messages (`url`). The router in step 14 will read the cards of all our agents and pick one per question; it never needs to know they're LangGraph graphs.

In [4]:
card_url = langgraph_card_url(AGENT_URL, "okf_agent")
card = httpx.get(card_url).json()
print(card_url, "\n")
print("name:        ", card["name"])
print("description: ", card["description"])
print("send to:     ", card["url"])
print("streaming:   ", card["capabilities"]["streaming"])
print("skills:      ", [s["name"] for s in card["skills"]])

http://127.0.0.1:2024/.well-known/agent-card.json?assistant_id=bb1780c6-cc9f-5dd1-a605-b5608e80a356 

name:         okf_agent
description:  Answers questions about the OKF bundle (GPT-4 technical report and related papers) from retrieved passages, with [concept id] citations. Replies 'The bundle does not contain this.' when the answer isn't in the bundle.
send to:      http://127.0.0.1:2024/a2a/bb1780c6-cc9f-5dd1-a605-b5608e80a356
streaming:    True
skills:       ['okf_agent Capabilities']


## 3. One A2A message, by hand

A2A is **JSON-RPC over HTTP**: one POST with a `method` and `params`. `SendMessage` waits until the agent is done (qwen's answer can take 10–60 s on your GPU).

The reply is a **task**:
- `status.state`: `TASK_STATE_COMPLETED`, or `TASK_STATE_FAILED` with a reason
- `artifacts`: the answer
- `contextId`: the **conversation**. Send it back with the next message for a follow-up.
- `history`: the messages so far in this conversation

In [5]:
request = {
    "jsonrpc": "2.0", "id": 1, "method": "SendMessage",
    "params": {"message": {"role": "ROLE_USER", "messageId": "msg-1",
                           "parts": [{"text": "What score did GPT-4 get on the bar exam?"}]}},
}
response = httpx.post(card["url"], json=request, timeout=300).json()

task = response["result"]["task"]
print("state:     ", task["status"]["state"])
print("contextId: ", task["contextId"])
print("answer:    ", task["artifacts"][0]["parts"][0]["text"])
print("\nthe raw response, shortened:")
print(json.dumps(response, indent=1)[:1500])

state:      TASK_STATE_COMPLETED
contextId:  2b4f9634-19ae-4778-86cc-1786b1d532d1
answer:     GPT-4 scored 298/400 on the Uniform Bar Exam, which corresponds to the 90th percentile (top 10%) of human test takers. [topics/simulated-exams] [papers/gpt-4-technical-report/4-capabilities-part-1]

the raw response, shortened:
{
 "jsonrpc": "2.0",
 "id": 1,
 "result": {
  "task": {
   "kind": "task",
   "id": "2b4f9634-19ae-4778-86cc-1786b1d532d1:01a14e6b-a764-74a0-a54f-ec8c67746435",
   "contextId": "2b4f9634-19ae-4778-86cc-1786b1d532d1",
   "history": [
    {
     "kind": "message",
     "role": "ROLE_USER",
     "parts": [
      {
       "kind": "text",
       "text": "What score did GPT-4 get on the bar exam?"
      }
     ],
     "messageId": "msg-1",
     "taskId": "2b4f9634-19ae-4778-86cc-1786b1d532d1:01a14e6b-a764-74a0-a54f-ec8c67746435",
     "contextId": "2b4f9634-19ae-4778-86cc-1786b1d532d1"
    },
    {
     "kind": "message",
     "role": "ROLE_AGENT",
     "parts": [
      {
   

## 4. The same with the client: `okf_agent/a2a_client.py`

`A2AClient` wraps that POST so the router can write `agent.send(question)`. It only uses the A2A protocol, so it works with **any** A2A agent, not just ours.

```python
okf = A2AClient.from_card(card_url)                    # read the card, remember its url
r1 = okf.send("What score did GPT-4 get on the bar exam?")
r2 = okf.send("And GPT-3.5?", context_id=r1.context_id)   # same conversation
```

In [6]:
okf = A2AClient.from_card(card_url)

r1 = okf.send("What score did GPT-4 get on the bar exam?")
print(r1.state, "|", r1.text, "\n")

r2 = okf.send("And GPT-3.5?", context_id=r1.context_id)     # follow-up: same contextId
print(r2.state, "|", r2.text)
print("\nsame conversation:", r2.context_id == r1.context_id)

TASK_STATE_COMPLETED | GPT-4 scored 298/400 on the Uniform Bar Exam, which corresponds to the 90th percentile (top 10%) of human test takers. [topics/simulated-exams] [papers/gpt-4-technical-report/4-capabilities-part-1] 

TASK_STATE_COMPLETED | GPT-3.5 scored 213/400 on the Uniform Bar Exam, which corresponds to the 10th percentile (bottom 10%) of human test takers. [topics/simulated-exams] [papers/gpt-4-technical-report/4-capabilities-part-1]

same conversation: True


The follow-up worked because the server **remembered** turn 1 under that `contextId`, so `prepare` saw the previous question, exactly like `thread_id` in `11`.

**Rules for `contextId`:**
- Leave it out on the first message: the server creates the conversation.
- Send back the one you got for follow-ups.
- It must be a **UUID the server made**. Your own name like `"chat-1"` is rejected (next cell).

In [7]:
try:
    okf.send("Hi", context_id="chat-1")
except Exception as exc:
    print(type(exc).__name__, "→", exc)

A2AError → okf_agent: Failed to create run: Invalid thread ID


## 5. What got stored

**`qa_log`** (Postgres) gets a row per question, as in `11`. `thread_id` is now the A2A `contextId`, so you can find a whole conversation in pgAdmin:

In [8]:
with psycopg.connect(config.postgres_url) as conn:
    cur = conn.execute("""SELECT id, created_at, thread_id, followup, search_query, answer, total_ms
                          FROM qa_log WHERE thread_id = %s ORDER BY id""", (r1.context_id,))
    rows = pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])
rows

,id,created_at,thread_id,followup,search_query,answer,total_ms
0,13,2026-10-18 15:21:05.014072+05:30,7157e5e0-6503-4834-b859-ae934caeb5f4,False,What score did GPT-4 get on the bar exam?,"GPT-4 scored 298/400 on the Uniform Bar Exam, ...",5733.7
1,14,2026-10-18 15:21:12.376174+05:30,7157e5e0-6503-4834-b859-ae934caeb5f4,True,What score did GPT-4 get on the bar exam? And ...,GPT-3.5 scored 213/400 on the Uniform Bar Exam...,6311.7


**The chat memory** lives in the agent server, saved to the `.langgraph_api/` folder in the project, so a conversation survives restarting `langgraph dev`. Add this line to `.gitignore`: `.langgraph_api/`.

Why not Postgres for the memory too, like in `11`? `langgraph dev` is the **development** server. Plugging our own Postgres checkpointer into it is possible but still marked alpha. For running locally, its own memory is the reliable choice. When we go public we'll pick the serving setup (stage 6), and Postgres memory comes back with it. `qa_log`, the log you analyse, is in Postgres either way.

## 6. Seeing it run: LangSmith and Studio

- **LangSmith → Tracing → `okf-agent`**: each A2A message is a run with the same tree as in `11` (prepare → search → answer → check → log). Runs from the server have the conversation as their **thread**, so the Threads tab shows the A2A conversations.
- **LangGraph Studio** (optional, nice for learning): open
  `https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024`
  in Chrome or Edge. You see the graph drawn, can type a question, and watch each node light up and its state change. Studio runs in the browser but talks to **your** local server; nothing is uploaded except traces (if tracing is on). (Safari and Brave block the connection to localhost.)

## 7. When something is down

A2A reports failures as a **task with state `FAILED`** and a reason, not as a crash, so the router can tell the user instead of hanging. To see it: stop the MCP server window, run the next cell, then start the MCP server again.

In [9]:
r = okf.send("What is RLHF?")
print(r.state, "|", r.text or "(no text)")
print("ok:", r.ok)

TASK_STATE_COMPLETED | Reinforcement learning from human feedback (RLHF) is a method that fine-tunes a large language model (LLM) using human feedback to improve alignment criteria such as helpfulness, honesty, and harmlessness. It consists of three parts: a pre-trained language model to align, a reward model (RM) trained on human preferences to predict rankings of model outputs, and an RL algorithm—typically Proximal Policy Optimization (PPO)—that uses the reward model to fine-tune the LLM. InstructGPT is a classic example, and GPT-4 was also fine-tuned with RLHF [topics/rlhf].
ok: True


## Where we are

- ✅ **10** okf_mcp: the bundle as MCP tools
- ✅ **11** okf_agent: LangGraph graph, Postgres memory and `qa_log`, LangSmith tracing
- ✅ **12** okf_agent **as an A2A service** (`langgraph dev`, agent card, `contextId` conversations)
- ⏭ **13** paper scout: a second agent that searches arXiv through a ready-made arXiv MCP server
- ⏭ **14** router: reads the agent cards and sends each question to the right agent over A2A
- ⏭ **15** report agent: turns answers into a report file (filesystem MCP server)